### SQL Query

In [ ]:
# SQL Query 0: Merge the three external data sources (literacy, health, aid) into the main panel
# WHAT: Adds female_literacy_pct, health_exp_pct, urban_pct, and oda_received to final_df
#       by matching every (iso3, year) row in final_df to the corresponding row in each
#       auxiliary table.
# HOW : Loads all four pandas DataFrames into an in-memory SQLite database, then defines
#       a VIEW (final_view) that performs three LEFT JOINs in a single statement, so the
#       join logic is written once and can be reused or re-queried. 
# WHY : Consolidating control variables into one table is required before the regression
#       and urgency-index modeling steps. 

conn = sqlite3.connect(":memory:")

final_df.to_sql("final",    conn, if_exists="replace", index=False)
literacy_wide.to_sql("lit", conn, if_exists="replace", index=False)
health_wide.to_sql("hlt",   conn, if_exists="replace", index=False)
aid_wide.to_sql("aid",      conn, if_exists="replace", index=False)

# Use a VIEW so the three joins are defined just once
conn.execute("""
    CREATE VIEW final_view AS
    SELECT f.*,
           lit.female_literacy_pct,
           hlt.health_exp_pct,
           hlt.urban_pct,
           aid.oda_received
    FROM final AS f
    LEFT JOIN lit ON f.iso3 = lit.iso3 AND f.year = lit.year
    LEFT JOIN hlt ON f.iso3 = hlt.iso3 AND f.year = hlt.year
    LEFT JOIN aid ON f.iso3 = aid.iso3 AND f.year = aid.year
""")

final_df = pd.read_sql("SELECT * FROM final_view", conn)
conn.close()

final_df.shape

In [ ]:
# SQL Query 1: Dataset scope check
# WHAT: Returns a one-row summary of the panel, which is number of distinct countries, earliest
#       and latest year covered, and total row count in wash_main.
# HOW : Uses aggregate functions (COUNT(DISTINCT country), MIN(year), MAX(year),
#       COUNT(*)) with no GROUP BY so SQLite returns exactly one summary row.
# WHY : A standard first-look sanity check. Confirms the panel's dimensions before any
#       modeling and immediately surfaces issues such as a wrong year filter, duplicated
#       rows, or unexpectedly low country coverage.
q1 = """
SELECT COUNT(DISTINCT country) AS num_countries,
       MIN(year)               AS earliest_year,
       MAX(year)               AS latest_year,
       COUNT(*)                AS total_rows
FROM wash_main
"""
pd.read_sql(q1, conn)

In [ ]:
# SQL Query 2: Countries with the most missing WASH data
# WHAT: Lists the top 10 countries by total number of missing observations across the
#       three core WASH indicators (improved water, basic handwashing, basic sanitation).
# HOW : For each indicator, uses SUM(CASE WHEN col IS NULL THEN 1 ELSE 0 END) to count
#       NULLs; the three counts are then summed inside the SELECT to produce a single
#       missing-value score per country. GROUP BY country/region rolls observations up to
#       the country level, and ORDER BY ... DESC LIMIT 10 surfaces the worst offenders.
# WHY : Reinforces the missing-value analysis from the EDA section and documents which
#       countries are most likely to drop out of the regression sample due to incomplete
#       WASH coverage. This is important context for interpreting which countries the final
#       urgency index can and cannot speak to.
q2 = """
SELECT country, region,
       SUM(CASE WHEN improved_water_pct IS NULL THEN 1 ELSE 0 END) +
       SUM(CASE WHEN basic_handwashing_pct IS NULL THEN 1 ELSE 0 END) +
       SUM(CASE WHEN basic_sanitation_pct IS NULL THEN 1 ELSE 0 END) AS missing_wash_count
FROM wash_main
GROUP BY country, region
ORDER BY missing_wash_count DESC
LIMIT 10
"""
pd.read_sql(q2, conn)

In [ ]:
# SQL Query 3: Average WASH indicators by region
# WHAT: Computes regional means of improved water access, basic handwashing, basic
#       sanitation, and household-level open defecation.
# HOW : Straightforward GROUP BY region with AVG over each indicator. Results are
#       rounded to two decimals for readability and sorted by water access ascending so
#       the worst-served regions appear first.
# WHY : Reveals where WASH deprivation is geographically concentrated. The expectation
#       (and a core motivation for this project) is that sub-Saharan Africa and South
#       Asia will show the lowest averages, which sets up the urgency-index analysis and
#       provides a regional benchmark used by later queries (e.g. Query 6).
q3 = """
SELECT region,
       ROUND(AVG(improved_water_pct), 2)     AS avg_improved_water,
       ROUND(AVG(basic_handwashing_pct), 2)  AS avg_handwashing,
       ROUND(AVG(basic_sanitation_pct), 2)   AS avg_sanitation,
       ROUND(AVG(open_defecation_hh_pct), 2) AS avg_open_defecation
FROM wash_main
GROUP BY region
ORDER BY avg_improved_water ASC
"""
pd.read_sql(q3, conn)

In [ ]:
# SQL Query 4: Average health outcomes by region (2020–2024)
# WHAT: Regional averages of diarrhea rate, infant mortality, and under-5 mortality for
#       the most recent five-year window. Units: deaths per 1,000 children.
# HOW : WHERE clause restricts to year BETWEEN 2020 AND 2024 and drops rows where the two
#       mortality columns are NULL (so the mean is not biased by countries missing the
#       outcome). AVG over each metric is grouped by region and ORDER BY u5_mortality DESC
#       puts the highest-burden regions on top.
# WHY : Complements Query 3 directly: if regions with low WASH access also show the
#       highest child mortality, that descriptive correlation motivates the formal
#       WASH-on-health regression and the construction of the urgency index. The 2020–2024
#       window is used to reflect the current burden rather than long-run averages.
q4 = """
SELECT
    region,
    ROUND(AVG(diarrhea_rate), 2) AS avg_diarrhea,
    ROUND(AVG(infant_mortality), 2) AS avg_infant_mortality,
    ROUND(AVG(u5_mortality), 2) AS avg_u5_mortality
FROM wash_main
WHERE year BETWEEN 2020 AND 2024
    AND infant_mortality IS NOT NULL
    AND u5_mortality IS NOT NULL
GROUP BY region
ORDER BY AVG(u5_mortality) DESC
"""
pd.read_sql(q4, conn)

In [ ]:
# SQL Query 5: Countries with under-5 mortality above the global average (2020–2024)
# WHAT: Returns the 15 countries whose average u5 mortality exceeds the global mean over
#       the 2020–2024 window.
# HOW : Uses a scalar subquery in the WHERE clause —
#           u5_mortality > (SELECT AVG(u5_mortality) FROM wash_main WHERE year BETWEEN ...)
#       The threshold is computed dynamically from the data rather than hard-coded.
#       The outer query then GROUP BYs country/region and orders DESC.
# WHY : Pre-screens the highest-burden countries before the urgency index is built. Also
#       acts as a sanity check downstream: the eventual top-ranked countries from the
#       index should overlap heavily with this list, since real mortality is one of the
#       inputs to urgency.
q5 = """
SELECT
    country,
    region,
    ROUND(AVG(u5_mortality), 2) AS avg_u5_mortality
FROM wash_main
WHERE year BETWEEN 2020 AND 2024
  AND u5_mortality > (
      SELECT AVG(u5_mortality)
      FROM wash_main
      WHERE year BETWEEN 2020 AND 2024
  )
GROUP BY country, region
ORDER BY avg_u5_mortality DESC
LIMIT 15
"""
pd.read_sql(q5, conn)

In [ ]:
# SQL Query 6: Countries below their own regional average in water access
# WHAT: Lists the 15 countries whose 2020–2024 improved-water-access average sits below
#       the average of their own region.
# HOW : JOINs wash_main against a derived table (the subquery aliased `reg`) that computes
#       the average water-access percentage per region. The outer query groups by
#       country/region and uses HAVING — not WHERE — to compare each country's aggregated
#       average against the region average, because the comparison involves an aggregate
#       and so must be evaluated after GROUP BY.
# WHY : Globally low countries are the obvious targets, but countries lagging their
#       regional peers expose localized policy gaps where the marginal return to WASH
#       investment is likely highest. Peers in similar geographic/economic contexts have
#       already demonstrated what is achievable, so the gap is not structural.
q6 = """
SELECT
    w.country,
    w.region,
    ROUND(AVG(w.improved_water_pct), 2) AS avg_water_pct,
    ROUND(reg.avg_water, 2) AS region_avg_water
FROM wash_main w
JOIN (
    SELECT
        region,
        AVG(improved_water_pct) AS avg_water
    FROM wash_main
    WHERE year BETWEEN 2020 AND 2024
      AND improved_water_pct IS NOT NULL
    GROUP BY region
) reg
ON w.region = reg.region
WHERE w.year BETWEEN 2020 AND 2024
  AND w.improved_water_pct IS NOT NULL
GROUP BY w.country, w.region, reg.avg_water
HAVING AVG(w.improved_water_pct) < reg.avg_water
ORDER BY avg_water_pct ASC
LIMIT 15
"""
pd.read_sql(q6, conn)

In [ ]:
# SQL Query 7: Within-region ranking of countries by under-5 mortality
# WHAT: Ranks every country against the other countries in its own region by average u5
#       mortality, with rank 1 = worst-performing in the region.
# HOW : Uses the RANK() window function with PARTITION BY region ORDER BY AVG(u5_mortality)
#       DESC, layered on top of a GROUP BY country/region. Because the ORDER BY inside
#       OVER() references an aggregate, the ranking is computed on the per-country averages
#       — no self-join needed.
# WHY : A country can look only moderately bad globally yet be the worst within its
#       region; that pattern is a strong signal for *region-targeted* intervention. This
#       view complements the global ranking in Query 5 by surfacing localized priorities
#       that a purely global cutoff would hide.
q7 = """
SELECT
    country,
    region,
    ROUND(AVG(u5_mortality), 2) AS avg_u5_mortality,
    RANK() OVER (
        PARTITION BY region
        ORDER BY AVG(u5_mortality) DESC
    ) AS rank_in_region
FROM wash_main
WHERE year BETWEEN 2020 AND 2024
  AND u5_mortality IS NOT NULL
GROUP BY country, region
ORDER BY region, rank_in_region
"""
pd.read_sql(q7, conn)

In [ ]:
# SQL Query 8: Annual and cumulative running average of improved water access
# WHAT: For each year in the panel, returns the cross-country average improved-water-access
#       percentage and a running average from the earliest year through that year.
# HOW : Nests a window function around the aggregate — AVG(AVG(improved_water_pct))
#       OVER (ORDER BY year). The inner AVG aggregates within each year (after GROUP BY
#       year); the outer AVG(...) OVER (ORDER BY year) accumulates those yearly means in
#       chronological order, giving a running mean without a self-join.
# WHY : Tracks whether global WASH conditions are improving over time. A rising running
#       average is consistent with sustained investment paying off at the global level;
#       a flat trajectory signals stagnation despite continued aid flows — and motivates
#       the country-level urgency analysis that follows.
q8 = """
SELECT year,
       ROUND(AVG(improved_water_pct), 2) AS avg_water_pct,
       ROUND(AVG(AVG(improved_water_pct)) OVER (ORDER BY year), 2) AS running_avg_water
FROM wash_main
GROUP BY year
ORDER BY year
"""
pd.read_sql(q8, conn)

In [ ]:
# SQL Query 9: Top 20 countries by the constructed Urgency Index
# WHAT: Returns the 20 highest-urgency countries with their rank, region, urgency score,
#       and urgency-level category (Critical / High / Moderate / Low).
# HOW : Simple SELECT ... ORDER BY rank LIMIT 20 from the precomputed urgency_results
#       table — that table was produced upstream in Python by combining WASH gaps,
#       mortality, and the regression-derived weights. Urgency_index is rounded to 2
#       decimals for the report.
# WHY : This is the headline deliverable of the entire analysis — the policy-relevant
#       list answering "where should WASH investment be directed first?". Restricting to
#       20 keeps the table readable while still capturing the most urgent cases.
q9 = """
SELECT rank, country, region,
       ROUND(urgency_index, 2) AS urgency_index,
       urgency_level
FROM urgency_results
ORDER BY rank
LIMIT 20
"""
pd.read_sql(q9, conn)

In [ ]:
# SQL Query 10: Validation — urgency level vs. actual under-5 mortality
# WHAT: For each urgency-level category, reports the number of countries assigned to it,
#       the average urgency index within the category, and the average *observed* under-5
#       mortality.
# HOW : GROUP BY urgency_level with COUNT(*) for category size and AVG over both the
#       index and the observed mortality. ORDER BY avg_urgency DESC puts the most urgent
#       group at the top so the comparison reads top-down.
# WHY : This is the construct-validation step for the index. If higher urgency tiers also
#       have systematically higher *actual* child mortality, then the categorical labeling
#       is consistent with the underlying health reality — which supports the index's
#       credibility for policy use. A weak or inverted gradient here would be a red flag.
q10 = """
SELECT urgency_level,
       COUNT(*)                      AS num_countries,
       ROUND(AVG(urgency_index), 2)  AS avg_urgency,
       ROUND(AVG(u5_mortality), 2)   AS avg_u5_mortality
FROM urgency_results
GROUP BY urgency_level
ORDER BY avg_urgency DESC
"""
pd.read_sql(q10, conn)